<a href="https://colab.research.google.com/github/jim20020317-lab/hw8-chiparch-reflection-agent/blob/master/HW8_AI_Agent_Reflection_%E6%99%B6%E7%89%87%E6%9E%B6%E6%A7%8B%E7%B4%85%E9%9A%8A%E5%AF%A9%E6%9F%A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠【AI Agent 作業】ChipArch Red-Team Reflection Agent
## 🔬 AI 晶片與小晶片 (Chiplet) 硬體架構紅隊反思優化系統

> **學校**：國立中央大學（中大）　|　**學號**：114521045　|　**系級**：電子組碩二　|　**姓名**：楊荏鈞
> **課程**：生成式 AI 實作課程　|　**作業**：第八週作業 —— 用程式做一個簡單的 AI Agent
> **選用 Agent 類型**：**Reflection（自我反思代理）**

---
### 📌【作業重點註明 1：任務目標與 Agent 類型選擇】

> **1. 任務核心背景**：
> 在高效能 AI 晶片與多小晶片（Multi-Chiplet / DCIM）架構設計中，初級工程師構思方案時常陷入「過度樂觀」的假設——只關注表面上的理論算力（TOPS），卻忽略底層電路物理極限（SRAM Subarray RC 延遲、Wordline/Bitline 過長造成的時序違例、讀寫衝突）、片上網路（NoC）頻寬記憶體牆與跨 Chiplet D2D 封裝能耗。
>
> **2. 選擇的 Agent 方法**：
> 本作業選擇 **Reflection（自我反思代理）**，參考老師【Demo07a】的設計理念進行深度原創改造，打造「初級設計師提案」➔「資深首席架構師紅隊審查」➔「架構師深度自我反思與工程修正」的完整閉環。
>
> **3. 預期達成效益**：
> 讓 AI 先產生第一版直覺架構，再自行進行多維度物理檢驗與批判，最後反思改寫出具備工程落地可行性的第二版優化架構。

### 📌【作業重點註明 2：Reflection 三階段閉環架構流程】

本系統在程式中具體呈現如下三階段 Reflection 流程：

```mermaid
flowchart TD
    User([👤 使用者晶片需求]) --> A[💡 階段一: Designer Agent 初級設計師]
    A -->|產生第一版架構初稿| B[🚨 階段二: Red-Team Reviewer 紅隊審查員]
    B -->|指出 3 大底層物理/時序/頻寬致命破綻| C[✨ 階段三: Refiner Agent 反思修正師]
    C -->|自我反思檢討 + 工程防護設計 + 量化對比表| Out([🖥️ 第二版可落地架構方案])
```

- **階段一（初稿提案）**：`Designer Agent` 產出初步 PE 陣列、數據流（Weight-Stationary / Output-Stationary）與互連拓撲。
- **階段二（紅隊批判審查）**：`Red-Team Reviewer` 直擊底層三大硬體致命傷（SRAM RC 延遲、NoC 死鎖風險、D2D 封裝頻寬瓶頸、封裝熱熱點）。
- **階段三（自我反思與優化）**：`Refiner Agent` 進行深度的 **Self-Reflection**，誠實檢討第一版盲點，給出工程落地防護方案，並輸出改善量化對照表。

### 1. 安裝套件並設定金鑰
安裝課程指定之 `aisuite[all]` 套件與 `gradio`，並讀取 API 金鑰（優先使用 Groq 服務）。

In [ ]:
!pip -q install "aisuite[all]" gradio

In [ ]:
import os

# 讀取 API 金鑰（支援 Colab Secrets 或環境變數）
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('Groq')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('GROQ_API_KEY', '')

if api_key:
    os.environ['GROQ_API_KEY'] = api_key
    print("🔑 Groq API 金鑰已成功載入！")
else:
    print("⚠️ 提示：若在 Colab 執行，請於左側『密鑰 (Secrets)』新增 'Groq' 金鑰。")

### 2. 設定 AISuite 與標準回應函式
選用 Groq 提供之旗艦開源模型 `groq:openai/gpt-oss-120b`（與老師 Demo07 完全一致），確保極速推論與頂尖的繁體中文工程分析能力。

In [ ]:
import aisuite as ai

provider = "groq"
model = "openai/gpt-oss-120b"

def reply(system, prompt, temperature=0.3):
    """標準 AISuite 呼叫函式"""
    client = ai.Client()
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=f"{provider}:{model}",
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

print(f"✅ AISuite 核心就緒，使用模型：{provider}:{model}")

### 📌【作業重點註明 3：三位 Agent 人設與分工定位】

> **1. Designer Agent（初版設計師）**：扮演自信樂觀的初級架構師，產出第一版初步提案。
> **2. Red-Team Reviewer Agent（紅隊審查員）**：扮演經驗豐富的資深架構師，專挑底層電路物理與拓撲漏洞，給出極度嚴苛的批判。
> **3. Refiner Agent（反思修正師）**：吸收審查意見進行深度 Self-Reflection，進行工程落地修正並產出對照表。

In [ ]:
# 1. 初級架構設計師人設
system_designer = """你是一位熱情且富有創意的「AI 晶片硬體架構初級設計師」。
當使用者提出一個 AI 模型加速或晶片規格需求時，請提出初步的硬體架構方案（第一版初稿）。
內容需包含：
1. 核心架構概述（PE 陣列、記憶體配置、計算數據流）
2. 預期效能與吞吐量分析
3. 互連拓撲設計（NoC、D2D 或晶圓級匯流排）
請使用台灣習慣的繁體中文，行文自信樂觀，條理分明。"""

# 2. 資深紅隊審查員人設
system_reviewer = """你是一位在頂尖半導體與 AI 晶片廠任職超過 20 年的「資深首席晶片架構師兼紅隊審查員（Red-Team Reviewer）」。
你的任務是針對初級設計師提交的硬體架構初稿進行極度嚴格、犀利且直擊痛點的【批判性反思與可行性審查】。
請著重於找出初稿中過度樂觀、忽略物理極限或隱藏死鎖的致命缺陷，並給出具體改進指標：
1. 【電路物理極限】：SRAM Subarray 的 RC 延遲、Wordline/Bitline 過長、讀寫衝突、記憶體牆。
2. 【互連與通信瓶頸】：NoC 拓撲是否會發生死鎖？跨 Chiplet 的 D2D 封裝頻寬是否過度承載？Partial Sum (psum) 是否跨晶片搬移造成巨額能耗？
3. 【散熱與面積 PPA 權衡】：封裝熱點（Thermal Hotspot）、晶片面積與良率折損。
請條列出 3 大關鍵致命漏洞，並給出明確、具工程可行性的修改指引。請用台灣繁體中文回應。"""

# 3. 反思優化架構師人設
system_refiner = """你是一位謙遜、注重工程嚴謹性並具備深度自我反思能力的「資深 AI 晶片架構修正師」。
你剛剛閱讀了資深紅隊審查員針對你第一版提案的嚴格批評與建議。
你的任務是進行深刻的【自我反思（Self-Reflection）】，徹底摒棄第一版中過度理想化的假設，並結合審查員的具體建議產出「第二版優化架構方案」。
輸出內容需包含：
1. 【反思自我檢討】：誠實點出第一版在底層物理與通訊網路上的盲點。
2. 【具體架構改進設計】：針對審查員指出的 3 大痛點提出具體解決方案（例如引入 Ping-Pong Relay Buffer、兩階段 BQP 映射、WS 權重原地保留）。
3. 【改進效益量化對比表】：以 Markdown 表格對比第一版與第二版在延遲、頻寬壓力、散熱風險與工程可行性上的具體改善。
請用台灣習慣的繁體中文回應。"""

print("✅ 三大 Agent 人設設定完成！")

### 📌【作業重點註明 4：Reflection 執行排程與 Agent 行為調節】

> **1. 完整閉環排程**：
> `reflect_architecture()` 依序觸發 Step 1（產生初稿）➔ Step 2（紅隊審查）➔ Step 3（反思改寫），三者緊密串聯，前後具備強烈的因果與邏輯關係。
>
> **2. 行為改良機制（9~10 分要件）**：
> 加入 `focus_mode` 參數（標準 PPA、極致低能耗、NoC 防死鎖），動態控制 Reviewer 的批判視角，讓 Agent 反思行為更加彈性可控。

In [ ]:
def reflect_architecture(user_query, focus_mode="⚖️ 標準 PPA 平衡審查"):
    """
    Reflection 核心流程：
    Step 1: Designer Agent 生成初稿
    Step 2: Red-Team Reviewer 進行批判性審查
    Step 3: Refiner Agent 吸收審查意見，進行自我反思並重寫為第二版
    """
    if not user_query.strip():
        return "請輸入有效的晶片設計需求！", "", ""

    # Step 1: Designer 生成第一版方案
    prompt_step1 = f"使用者晶片設計需求：{user_query}\n請針對此需求設計初步硬體加速架構方案。"
    first_version = reply(system_designer, prompt_step1)

    # Step 2: Reviewer 進行紅隊批判與反思檢驗
    prompt_step2 = f"審查重點取向：{focus_mode}\n\n這是初級設計師提出的架構提案：\n{first_version}\n\n請以資深架構師的紅隊視角進行嚴厲審查，點出三大致命破綻並給予具體工程改善要求。"
    suggestion = reply(system_reviewer, prompt_step2)

    # Step 3: Refiner 根據批判進行 Reflection，改進並產出第二版
    prompt_step3 = f"這是你的第一版提案：\n{first_version}\n\n這是紅隊審查員的犀利批評與改進建議：\n{suggestion}\n\n請進行深刻的【自我反思（Self-Reflection）】，修正錯誤假設，產出第二版優化架構方案與改善對照表。"
    second_version = reply(system_refiner, prompt_step3)

    return first_version, suggestion, second_version

### 📌【作業重點註明 5：Gradio 三欄位對比視覺化 WebApp】

> **介面設計特點**：
> 1. **三欄同步對照**：將【第一版初稿】、【紅隊審查批判】與【第二版反思優化】並排展示，使用者一目了然 Reflection 的改進軌跡。
> 2. **內建四組真實晶片架構案例**：包含 8-Chiplet 權重廣播、SRAM DCIM 跨層流水線、晶圓級晶片環狀拓撲、以及貪婪任務分配，點擊即可一鍵測試。

In [ ]:
import gradio as gr

# 預設晶片架構測試案例
example_cases = [
    [
        "我想設計一個支援 70B 大模型推論的 8-Chiplet 加速系統，第一版打算在所有 Chiplet 間採用全廣播（Broadcast）同步每一層的權重與中間特徵圖，以最大化計算並行度。",
        "⚖️ 標準 PPA 平衡審查"
    ],
    [
        "我想在 SRAM DCIM（數位存算一體）Macro 內部直接切跨層流水線（Interlayer Pipelining），以減少層間乒乓搬移並壓縮晶片面積。",
        "⚡ 極致低能耗與封裝散熱審查"
    ],
    [
        "設計一個晶圓級晶片（Wafer-Scale）的 CNN 加速架構，在數萬個 PE 間採用簡單的單向環狀拓撲傳遞 Partial Sum，不使用額外的 Relay Buffer。",
        "🚀 片上網路頻寬與防死鎖審查"
    ],
    [
        "多小晶片加速器中，直接採用貪婪演算法（Greedy Search）動態分配各層任務節點，以期達到最佳即時負載平衡。",
        "⚖️ 標準 PPA 平衡審查"
    ]
]

custom_css = """
body, .gradio-container {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
    background-color: #f8fafc;
}
.header-card {
    background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%);
    color: white;
    padding: 24px;
    border-radius: 12px;
    margin-bottom: 20px;
    box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
}
.header-card h1 {
    color: #38bdf8;
    margin-bottom: 6px;
    font-size: 26px;
}
.header-card p {
    color: #cbd5e1;
    font-size: 14px;
    margin: 4px 0;
}
"""

with gr.Blocks(title="ChipArch Red-Team Reflection Agent", css=custom_css) as demo:
    gr.HTML("""
    <div class="header-card">
        <h1>🧠 ChipArch Red-Team Reflection Agent</h1>
        <p><strong>專題主題</strong>：AI 晶片與小晶片 (Chiplet) 設計提案之自我反思優化系統（Reflection Agent 實作）</p>
        <p><strong>學生資訊</strong>：中央大學 電子組碩二 114521045 楊荏鈞 ｜ <strong>課程</strong>：生成式 AI 實作作業（AI Agent）</p>
        <p style="font-size: 13px; color: #93c5fd; margin-top: 8px;">
            🔄 Agent 運作機制：【初級架構師提案】➔【資深首席架構師紅隊批判反思】➔【反思優化與工程落地版方案】
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=8):
            user_input = gr.Textbox(
                label="💡 請輸入您的 AI 晶片或小晶片硬體架構構想：",
                placeholder="例如：我想設計一個 8-Chiplet 系統，第一版打算在所有晶片間全廣播權重以最大化並行度...",
                lines=3
            )
        with gr.Column(scale=4):
            focus_mode = gr.Radio(
                choices=["⚖️ 標準 PPA 平衡審查", "⚡ 極致低能耗與封裝散熱審查", "🚀 片上網路頻寬與防死鎖審查"],
                value="⚖️ 標準 PPA 平衡審查",
                label="🎯 紅隊反思審查焦點（Agent 行為調整）："
            )
            submit_btn = gr.Button("🚀 啟動 Reflection Agent 反思推論", variant="primary", size="lg")

    gr.Markdown("### 🔍 點擊以下推薦晶片架構提案進行快速測試：")
    gr.Examples(examples=example_cases, inputs=[user_input, focus_mode])

    gr.Markdown("---")
    gr.Markdown("## 🔄 Reflection 三階段推論進程展示")

    with gr.Row():
        out1 = gr.Textbox(
            label="💡 [階段一] Designer Agent 初版提案（未經反思的直覺構想）",
            lines=16,
            show_copy_button=True
        )
        out2 = gr.Textbox(
            label="🚨 [階段二] Red-Team Reviewer 批判審查（直擊物理/時序/頻寬致命缺陷）",
            lines=16,
            show_copy_button=True
        )
        out3 = gr.Textbox(
            label="✨ [階段三] Refiner Agent 反思優化成果（自我反思檢討＋具體工程修正＋改善對比表）",
            lines=16,
            show_copy_button=True
        )

    submit_btn.click(
        reflect_architecture,
        inputs=[user_input, focus_mode],
        outputs=[out1, out2, out3]
    )

# 啟動 Gradio 互動介面（支援公開分享連結 share=True）
demo.launch(share=True, debug=True)